# Exportar Data Mart a Parquet

Ejecuta este cuadernillo completo. Generará `data/datamart_accidentes.parquet`, que luego subirás a GitHub junto con `streamlit_app.py` y `requirements.txt`.

In [1]:

# Exportar Data Mart a Parquet para despliegue en Streamlit Cloud
# Ejecuta este cuadernillo completo. Al finalizar se crea: data/datamart_accidentes.parquet

import os
import sys
import subprocess
import importlib.util
from pathlib import Path

# Instalación automática si faltan paquetes en tu entorno de Jupyter
paquetes = {
    "pyodbc": "pyodbc",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])

import pyodbc
import pandas as pd

# Credenciales tomadas del cuadernillo poblar_datamart(1).ipynb / dashboards(1).ipynb
DRIVER = "ODBC Driver 17 for SQL Server"
SERVER = "LAPTOP-RSS8LMV8"
DATABASE = "proyecto_destino"
UID = "sa"
PWD = "@DDJJrrvv1303"

CONN_STR = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={UID};"
    f"PWD={PWD};"
    "TrustServerCertificate=yes;"
)

print("Conectando a SQL Server...")
conn = pyodbc.connect(CONN_STR, timeout=30)
print("Conexión exitosa")

query = """
SELECT
    fa.id,
    fa.duracion_minutos,
    fa.distance,
    fa.temperature,
    fa.humidity,
    fa.visibility,
    fa.wind_speed,
    fa.precipitation,

    ds.severity,

    df.start_time,
    df.anio,
    df.mes,
    df.dia,
    df.hora,
    df.dia_semana,
    df.es_fin_semana,

    dl.street,
    dl.city,
    dl.county,
    dl.state,
    dl.zipcode,
    dl.timezone,

    dc.weather_condition,
    dct.civil_twilight,

    dcr.crossing,
    dj.junction,
    dst2.station,
    dsp.stop,
    dtr.traffic_signal

FROM FactAccidente fa
JOIN DimSeverity      ds   ON fa.severityID      = ds.id
JOIN DimFecha         df   ON fa.fechaID         = df.id
JOIN DimLugar         dl   ON fa.lugarID         = dl.id
JOIN DimClima         dc   ON fa.climaID         = dc.id
JOIN DimCivilTwilight dct  ON fa.civilTwilightID = dct.id
JOIN DimCrossing      dcr  ON fa.crossingID      = dcr.id
JOIN DimJunction      dj   ON fa.junctionID      = dj.id
JOIN DimStation       dst2 ON fa.stationID       = dst2.id
JOIN DimStop          dsp  ON fa.stopID          = dsp.id
JOIN DimTraffic       dtr  ON fa.trafficID       = dtr.id
"""

print("Leyendo consulta del Data Mart...")
df = pd.read_sql(query, conn)
conn.close()
print(f"Registros leídos: {len(df):,}")

# Limpieza compatible con el dashboard
print("Preparando tipos de datos...")
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["severity"] = pd.to_numeric(df["severity"], errors="coerce").astype("Int64")

for col in ["crossing", "junction", "station", "stop", "traffic_signal"]:
    df[col] = df[col].map({"True": 1, "False": 0, True: 1, False: 0, "1": 1, "0": 0}).fillna(0).astype(int)

numeric_cols = [
    "duracion_minutos", "distance", "temperature", "humidity",
    "visibility", "wind_speed", "precipitation", "anio", "mes", "dia", "hora"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Exportación
Path("data").mkdir(exist_ok=True)
out = Path("data") / "datamart_accidentes.parquet"
df.to_parquet(out, index=False)

print("Exportación finalizada")
print(f"Archivo generado: {out.resolve()}")
print(f"Tamaño aproximado: {out.stat().st_size / (1024*1024):.2f} MB")

df.head()


Conectando a SQL Server...
Conexión exitosa
Leyendo consulta del Data Mart...


C:\Users\Diego\AppData\Local\Temp\ipykernel_10440\2668881536.py:95: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros leídos: 7,533,737
Preparando tipos de datos...
Exportación finalizada
Archivo generado: C:\Users\Diego\Documents\Business Intelligence\data\datamart_accidentes.parquet
Tamaño aproximado: 253.39 MB


,id,duracion_minutos,distance,temperature,humidity,visibility,wind_speed,precipitation,severity,start_time,...,state,zipcode,timezone,weather_condition,civil_twilight,crossing,junction,station,stop,traffic_signal
0,A-1,314.0,0.01,36.9,91.0,10.0,NaN,0.02,3,2016-02-08 05:46:00,...,OH,45424,US/Eastern,Light Rain,Night,0,0,0,0,0
1,A-10,30.0,0.01,37.4,100.0,3.0,4.6,0.02,3,2016-02-08 08:10:00,...,OH,43081-8940,US/Eastern,Light Rain,Day,0,0,0,0,0
2,A-100,30.0,0.01,7.5,87.0,10.0,4.6,NaN,2,2016-02-11 08:13:00,...,OH,45432,US/Eastern,Scattered Clouds,Day,0,0,0,0,0
3,A-1000,30.0,0.00,77.0,34.0,10.0,3.5,NaN,2,2016-06-23 10:31:00,...,CA,95762-6704,US/Pacific,Clear,Day,0,0,0,0,0
4,A-10000,29.0,0.01,46.0,71.0,10.0,8.1,NaN,3,2017-01-06 16:22:00,...,CA,95605,US/Pacific,Clear,Day,0,0,0,0,0
